# Develop multi-agent research assistant

In [1]:
from dotenv import load_dotenv
load_dotenv()

import os

from autogen_ext.models.openai import OpenAIChatCompletionClient


In [2]:
model_client = OpenAIChatCompletionClient(
    model="gpt-5-nano",
    api_key=os.getenv("OPENAI_API_KEY")
)

Import all libraries

In [4]:
from typing import List, Sequence
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.messages import BaseAgentEvent, BaseChatMessage
from autogen_agentchat.teams import SelectorGroupChat, RoundRobinGroupChat
from autogen_agentchat.ui import Console
import requests
import xml.etree.ElementTree as ET


In [ ]:
def arxiv_search_tool(term: str, max_results: int):
    url = "http://export.arxiv.org/api/query"
    search_query = f"all:{term}"
    start = 0
    max_results = max_results
    sort_by = "relevance"
    params = {
        "search_query": search_query,
        "start": start,
        "max_results": max_results,
        "sortBy": sort_by
        }
    response = requests.get(url, params=params)
    if response.status_code == 200:
        root = ET.fromstring(response.content)
        entries = root.findall("{http://www.w3.org/2005/Atom}entry")
        results = []
        for entry in entries:
            title = entry.find("{http://www.w3.org/2005/Atom}title").text
            summary = entry.find("{http://www.w3.org/2005/Atom}summary").text
            link = entry.find("{http://www.w3.org/2005/Atom}id").text
            results.append({"title": title, "summary": summary, "link": link})
        return results
    else:
        return f"Error: {response.status_code} - {response.text}"

